# Week 59, Day 3: Generator expressions

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w059d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
squares_list = [n * n for n in range(5)]
squares_gen = (n * n for n in range(5))
print(squares_list, type(squares_gen).__name__)
print(sum(squares_gen), sum(squares_gen))
print(sum(n * n for n in range(5)), max(len(w) for w in "the quick brown fox".split()))

In [ ]:
words = ["kettle", "Tea", "milk", "Teapot", "cup"]
print(sorted((w.lower() for w in words), reverse=True))
print(tuple(len(w) for w in words), {w[0].lower() for w in words})
print(list(f"{a}{b}" for a in "xy" for b in "12" if not (a == "y" and b == "1")))

In [ ]:
checked = []

def is_slow(n):
    checked.append(n)
    return n * n > 50

print(any(is_slow(n) for n in range(100)), checked)
print(next((w for w in ["ab", "abcdefghij", "xyz"] if len(w) > 8), None))
print(all(n % 2 == 0 for n in [2, 4, 5, 6]), next((w for w in [] if w), "none"))

In [ ]:
import itertools

source = ["  12", "# a comment", "", "-3", "40 ", "7"]
lines = (line.strip() for line in source)
filled = (line for line in lines if line and not line.startswith("#"))
numbers = (int(line) for line in filled)
print(sum(n for n in numbers if n > 0))

endless = itertools.count(1)
odd_squares = (n * n for n in endless if n % 2)
print(next(sq for sq in odd_squares if sq > 1000))

In [ ]:
limit = 3
small = (n for n in range(10) if n < limit)     # range(10) is evaluated now; "n < limit" isn't
limit = 6
print(list(small))                              # uses limit = 6

items = [1, 2, 3]
doubled = (x * 2 for x in items)                # the list object is captured now
items = [10, 20]                                # rebinding the name doesn't matter
print(list(doubled))

> **Key idea.** A generator expression is a comprehension in round brackets: lazy, one-shot, an iterator. As a sole argument its brackets can be dropped; otherwise it needs its own. any and all stop at the first decisive value and next takes the first match, so with a generator expression the rest are never computed; sum, max and friends see everything without storing it. Chained generator expressions form constant-memory pipelines where each value passes through every stage in turn. The first for's iterable is evaluated at once; everything else when values are pulled, so consume generators near where you make them. Use a list when you need the values again or need list operations, a generator expression for one pass, and a generator function when the logic needs statements.

### Exercise: One line each

Write each with a single generator expression (no list comprehensions, no loops written out): `total_cost(basket)`: the sum of `price * quantity` over `(name, price, quantity)` tuples; `has_negative(nums)`; `longest_word_length(text)` (0 for no words); and `first_vowel_word(words)`: the first word starting with a vowel, or `None`.

In [ ]:
def total_cost(basket):
    ___

def has_negative(nums):
    ___

def longest_word_length(text):
    ___

def first_vowel_word(words):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools\ndef test_onepass():\n    \"generator expressions\"\n    assert total_cost([(\"tea\", 2, 3), (\"jam\", 4, 1)]) == 10 and total_cost([]) == 0\n    assert has_negative([3, -1, 2]) and not has_negative([]) and longest_word_length(\"a bb ccc\") == 3 and longest_word_length(\"\") == 0\n    assert first_vowel_word([\"tree\", \"apple\", \"egg\"]) == \"apple\" and first_vowel_word([\"x\"]) is None\n    assert has_negative(itertools.chain([1, 2, -5], itertools.count()))\n    assert first_vowel_word(itertools.chain([\"dog\", \"owl\"], itertools.repeat(\"cat\"))) == \"owl\"\n    import re\n    assert not re.search(r\"\\[[^\\]]*\\bfor\\b\", source) and \"for \" in source and \"\\n    for\" not in source")

### Exercise: Counting in a stream

Write `count_with(lines, word)`: how many lines contain the word (case-insensitive, as a whole word among the line's words), and `average_length(lines)`: the mean number of characters per non-blank line (after stripping), rounded to 1 place, `0.0` if none: both in one pass over `lines`, which may be a one-shot iterator. Use generator expressions; for the average, count and total together in a single loop if you need to.

In [ ]:
def count_with(lines, word):
    ___

def average_length(lines):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_lines():\n    \"one pass\"\n    text = [\"The cat sat\", \"a dog\", \"\", \"CAT and cat\", \"concatenate\"]\n    assert count_with(text, \"cat\") == 2 and count_with(iter(text), \"dog\") == 1 and count_with([], \"x\") == 0\n    assert average_length(iter(text)) == round((11 + 5 + 11 + 11) / 4, 1) and average_length([\"\", \"  \"]) == 0.0")

### Exercise: Two for clauses

Write `pairs_with_sum(nums, target)`: a generator expression (returned, not consumed) of `(i, j)` index pairs with i < j and `nums[i] + nums[j] == target`, and `flatten(rows)`: a generator expression of every item of a list of lists, row by row. Then `first_pair(nums, target)`: the first such pair or `None`, without building all of them.

In [ ]:
def pairs_with_sum(nums, target):
    ___

def flatten(rows):
    ___

def first_pair(nums, target):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import types\ndef test_nested():\n    \"nested clauses\"\n    g = pairs_with_sum([1, 4, 2, 3], 5)\n    assert isinstance(g, types.GeneratorType) and list(g) == [(0, 1), (2, 3)]\n    assert list(flatten([[1, 2], [], [3]])) == [1, 2, 3] and isinstance(flatten([]), types.GeneratorType)\n    assert first_pair([1, 4, 2, 3], 5) == (0, 1) and first_pair([1, 2], 9) is None\n    assert first_pair(list(range(100_000)), 3) == (0, 3)")

### Exercise: A pipeline of stages

Write `clean_numbers(lines)`: a lazy pipeline (returned as a generator) that strips each line, drops blank lines and comments (starting with `#`), converts to `int`, and keeps only the even ones. Use one generator expression per stage. Then `running_max(numbers)`: a **generator function** yielding the largest value so far after each number (a running maximum needs state between items, so it can't be one expression).

In [ ]:
def clean_numbers(lines):
    ___

def running_max(numbers):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, types\ndef test_pipeline():\n    \"stages\"\n    src = [\" 4\", \"# x\", \"\", \"7\", \"10 \", \"-2\"]\n    assert list(clean_numbers(src)) == [4, 10, -2] and isinstance(clean_numbers(src), types.GeneratorType)\n    assert list(running_max([3, 1, 4, 1, 5])) == [3, 3, 4, 4, 5] and list(running_max([])) == []\n    endless = (str(n) for n in itertools.count())\n    assert list(itertools.islice(clean_numbers(endless), 3)) == [0, 2, 4]\n    assert list(itertools.islice(running_max(clean_numbers(str(n) for n in itertools.count())), 3)) == [0, 2, 4]")

### Exercise: Freezing a value

Write `below(limit, numbers)`: a **generator function** yielding the numbers less than limit, so that the limit is fixed when it's called. Then `make_filters(limits, numbers)`: a list of generators, one per limit, each yielding the numbers below that limit, built in a loop. (With generator expressions `(n for n in numbers if n < limit)` built in the same loop, every generator would see the **last** limit; with `below`, each keeps its own.)

In [ ]:
def below(limit, numbers):
    ___

def make_filters(limits, numbers):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_freeze():\n    \"values fixed at call time\"\n    gens = make_filters([2, 4, 6], [1, 3, 5, 7])\n    assert [list(g) for g in gens] == [[1], [1, 3], [1, 3, 5]]\n    assert list(below(0, [1, 2])) == [] and list(below(10, range(3))) == [0, 1, 2]")

### Bug hunt: Brackets for two arguments

A generator expression needs its own brackets when the call has another argument. Without them this is a syntax error. Add the brackets so it prints `['tea', 'milk', 'jam']`.

In [ ]:
items = ["Jam", "Tea", "Milk"]
print(sorted(w.lower() for w in items, reverse=True))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"own brackets\"\n    expect_output(\"['tea', 'milk', 'jam']\")")

### Bug hunt: A changed limit

The condition `n < limit` is evaluated when values are pulled, not when the generator is made, so after `limit` changes this prints `[0, 1, 2, 3, 4]` instead of the three numbers below the original limit. Consume the generator before the limit changes (build the list straight away) so it prints `[0, 1, 2]`.

In [ ]:
limit = 3
small = (n for n in range(10) if n < limit)
limit = 5
print(list(small))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"evaluated late\"\n    expect_output(\"[0, 1, 2]\")")

### Bug hunt: Checking everything first

`any([...])` builds the whole list before `any` looks at it, so the slow check runs for all 1,000 values and this prints `True 1000`. Use a generator expression so `any` stops at the first match: `True 4`.

In [ ]:
calls = [0]
def slow_check(n):
    calls[0] += 1
    return n > 2

print(any([slow_check(n) for n in range(1000)]), calls[0])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"stop early\"\n    expect_output(\"True 4\")")

### Puzzle: A three-stage pipeline

Arrange the lines of a lazy pipeline that totals the positive numbers in a list of strings, skipping blanks. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # type((x for x in [])).__name__
q = ___   # g = (x for x in range(3)); list(g) + list(g)
r = ___   # sum(1 for c in "banana" if c == "a")
s = ___   # next((x for x in [4, 9, 16] if x > 5), 0)
t = ___   # True or False: (x for x in data) makes a tuple

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"generator\" and q == [0, 1, 2] and r == 3, \"p, q and r\"\n    assert s == 9 and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a report in one pass

A till writes one line per sale: `"2026-11-07,tea,2.50"` (date, item, price), with blank lines and `#` comments mixed in, and possibly broken lines (wrong number of fields or a price that isn't a number), which must be **skipped**. Using generator expressions and generator functions (no list of all the sales), write:

- `sales(lines)`: a generator of `(date, item, price)` for the valid lines, `price` a float.
- `report(lines)`: a dict with `"count"`, `"total"` (rounded to 2 places), `"biggest"` (the `(date, item, price)` with the highest price, the first on a tie, `None` if no sales) and `"items"` (a dict from item to total spent, rounded to 2 places), computed in **one** pass over `lines` (they may come from a one-shot iterator).
- `first_big(lines, limit)`: the first sale with price over limit, or `None`, reading no further than needed (the checks give it an endless stream).

In [ ]:
def sales(lines):
    ___

def report(lines):
    ___

def first_big(lines, limit):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, types\nLINES = [\"# till 3\", \"2026-11-07,tea,2.50\", \"\", \"2026-11-07,cake,3.20\", \"broken line\", \"2026-11-08,tea,2.50\", \"2026-11-08,jam,x.yz\", \"2026-11-08,scone,3.20\"]\ndef test_report():\n    \"one pass\"\n    assert isinstance(sales(LINES), types.GeneratorType)\n    assert list(sales(LINES)) == [(\"2026-11-07\", \"tea\", 2.5), (\"2026-11-07\", \"cake\", 3.2), (\"2026-11-08\", \"tea\", 2.5), (\"2026-11-08\", \"scone\", 3.2)]\n    r = report(iter(LINES))\n    assert r == {\"count\": 4, \"total\": 11.4, \"biggest\": (\"2026-11-07\", \"cake\", 3.2), \"items\": {\"tea\": 5.0, \"cake\": 3.2, \"scone\": 3.2}}\n    assert report([]) == {\"count\": 0, \"total\": 0.0, \"biggest\": None, \"items\": {}}\n    def endless():\n        yield \"2026-01-01,tea,1.00\"\n        yield \"2026-01-01,cake,9.99\"\n        while True:\n            yield \"junk\"\n    assert first_big(endless(), 5) == (\"2026-01-01\", \"cake\", 9.99) and first_big(LINES, 100) is None")

## Warm-up

The generator expression is used up by the first `sum`, so the second gets nothing and returns 0. And when a generator expression is the **only** argument to a function, its own brackets can be dropped: `sum(n * n for n in range(5))`.

## Round brackets

A generator expression has exactly the syntax of a list comprehension, with round brackets:

`(expression for item in iterable if condition)`

with any number of `for` and `if` clauses, read left to right like nested loops. The result is a generator: one-shot, lazy, and an iterator, so it can go anywhere an iterable is expected.

**The bracket rule**: if a generator expression is the sole argument of a call, the call's brackets are enough: `sum(x for x in xs)`. With any other argument, it needs its own: `sorted((x for x in xs), reverse=True)`. Without them, Python reports a syntax error.

Comprehensions for other types follow the same pattern: `{k: v for ...}` builds a dict, `{x for ...}` a set, and `tuple(x for ...)` a tuple. There is no "tuple comprehension": round brackets mean a generator.

## Stopping early

The real gain of laziness: functions that can decide early **stop pulling values**, so the rest are never computed.

- `any(...)` stops at the first True, `all(...)` at the first False.
- `next(genexp, default)` takes just the first match: "the first word longer than 8 letters, or None".
- `sum`, `min`, `max`, `len` of a `list`... have to see everything, but still never store it: `max(len(line) for line in open_file)` reads a huge file one line at a time.

Only 0 to 8 were checked: `any` stopped at the first square over 50. A list comprehension inside `any([...])` would have checked all 100 first.

## Pipelines

Generator expressions can feed one another, each stage pulling from the one before:

```python
lines = (line.strip() for line in source)
filled = (line for line in lines if line and not line.startswith("#"))
numbers = (int(line) for line in filled)
total = sum(n for n in numbers if n > 0)
```

Nothing happens until the last line pulls values through. Each value travels through **every** stage before the next one starts, so the whole pipeline uses constant memory however long the input, and works on endless streams if the final step stops early. (David Beazley's talks "Generator Tricks for Systems Programmers", from 2008, made this style famous for processing logs; Thursday goes further with it.)

## When things are evaluated

One rule surprises almost everyone: in a generator expression, the **first** `for`'s iterable is evaluated **immediately**, when the expression is created, but everything else (the conditions, the expression, later `for` clauses) is evaluated **later**, when each value is pulled. So if a variable used inside changes before you consume the generator, the new value is used.

This is usually harmless, but it bites when a generator is created in one place and consumed later, especially in a loop that rebinds the variable it uses.

The safe habit: consume a generator expression close to where you create it, or pass the values it needs as arguments to a generator **function**, which freezes them at call time.

## Which to use

| Use | When |
|---|---|
| a list comprehension `[...]` | you need the values more than once, need `len`, indexing or sorting in place, or the data is small and speed matters (lists are a little faster to build and loop) |
| a generator expression `(...)` | you consume the values once, straight away, especially into `sum`, `any`, `all`, `max`, `next` or `join`, or the data is big or endless |
| a generator function | the logic needs statements: several steps, state between items, `try`, cleanup, or it's reused with different arguments |

A useful rule of thumb: if you write `[... for ...]` only to pass it straight into a function that loops over it once, drop the square brackets.

## Practice

## Quiz

## Daily challenge

## Before you go

Search your earlier exercises for `sum([`, `any([`, `all([`, `max([` and `"".join([`, and drop the square brackets in each. Tomorrow: `yield from` and longer pipelines.